In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Load the dataset
def load_data():
    # Assuming the NSL-KDD data is in CSV format, adjust the file paths accordingly
    train_data = pd.read_csv('KDDTrain+.txt')
    test_data = pd.read_csv('KDDTest+.txt')

    return train_data, test_data

# Preprocess data: convert categorical variables and normalize numeric features
'''def preprocess_data(df):
    # Encode categorical columns (protocol_type, service, flag)
    label_encoder = LabelEncoder()
    df['protocol_type'] = label_encoder.fit_transform(df['protocol'])
    df['service'] = label_encoder.fit_transform(df['service'])
    df['flag'] = label_encoder.fit_transform(df['flag'])

    # Features and labels
    X = df.drop(columns=['label'])  # Drop label column (target variable)
    y = df['label'].apply(lambda x: 1 if x == 'normal' else 0)  # Binary classification (normal = 0, attack = 1)

    # Normalize data (optional but recommended)
    X = (X - X.mean()) / X.std()

    return X, y''' 

# Preprocess data: convert categorical variables and normalize numeric features
def preprocess_data(df):
    print("Columns in the dataset:", df.columns)  # Debugging line to print column names
    required_columns = ['protocol_type', 'service', 'flag']
    
    # Check if required columns exist
    for col in required_columns:
        if col not in df.columns:
            print(f"Warning: Column {col} is missing.")
    
    # Encode categorical columns (protocol_type, service, flag)
    label_encoder = LabelEncoder()
    if 'protocol_type' in df.columns:
        df['protocol_type'] = label_encoder.fit_transform(df['protocol_type'])
    if 'service' in df.columns:
        df['service'] = label_encoder.fit_transform(df['service'])
    if 'flag' in df.columns:
        df['flag'] = label_encoder.fit_transform(df['flag'])

    # Features and labels
    X = df.drop(columns=['label'])  # Drop label column (target variable)
    y = df['label'].apply(lambda x: 1 if x == 'normal' else 0)  # Binary classification (normal = 0, attack = 1)

    # Normalize data (optional but recommended)
    X = (X - X.mean()) / X.std()

    return X, y


# Build the RNN model
def build_rnn(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=input_shape),
        tf.keras.layers.SimpleRNN(64, activation='relu', return_sequences=True),
        tf.keras.layers.SimpleRNN(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')  # Binary output (0 or 1)
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Plot confusion matrix
def plot_confusion_matrix(cm):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.title('Confusion Matrix')
    plt.show()

# Plot ROC Curve
def plot_roc_curve(fpr, tpr, auc_score):
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='b', label=f'ROC curve (area = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

# Plot Precision-Recall Curve
def plot_precision_recall_curve(precision, recall, thresholds):
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='b')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.show()

# Main function to execute all tasks
def main():
    # Load and preprocess data
    train_data, test_data = load_data()
    train_data.head(3)
    test_data.head(3)
    X_train, y_train = preprocess_data(train_data)
    X_test, y_test = preprocess_data(test_data)

    # Split into training, validation, and test sets
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    # Build and train the RNN model
    model = build_rnn(X_train.shape[1:])
    model.summary()

    # Train the model
    history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_val, y_val))

    # Evaluate the model on the test data
    y_pred = (model.predict(X_test) > 0.5).astype(int)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:\n", cm)
    plot_confusion_matrix(cm)

    # Classification report
    print("Classification Report:\n", classification_report(y_test, y_pred))

    # Compute ROC curve and AUC
    fpr, tpr, _ = roc_curve(y_test, model.predict(X_test))
    auc_score = auc(fpr, tpr)
    print(f"ROC AUC Score: {auc_score:.2f}")
    plot_roc_curve(fpr, tpr, auc_score)

    # Compute Precision-Recall curve
    precision, recall, thresholds = precision_recall_curve(y_test, model.predict(X_test))
    plot_precision_recall_curve(precision, recall, thresholds)

if __name__ == '__main__':
    main()


Columns in the dataset: Index(['0', 'tcp', 'ftp_data', 'SF', '491', '0.1', '0.2', '0.3', '0.4', '0.5',
       '0.6', '0.7', '0.8', '0.9', '0.10', '0.11', '0.12', '0.13', '0.14',
       '0.15', '0.16', '0.18', '2', '2.1', '0.00', '0.00.1', '0.00.2',
       '0.00.3', '1.00', '0.00.4', '0.00.5', '150', '25', '0.17', '0.03',
       '0.17.1', '0.00.6', '0.00.7', '0.00.8', '0.05', '0.00.9', 'normal',
       '20'],
      dtype='object')


KeyError: "['label'] not found in axis"